In [1]:
# ClickHouse Connection Setup
from clickhouse_driver import Client
import pandas as pd
import configparser
from pathlib import Path

# Load configuration from config file
config = configparser.ConfigParser()
config_path = Path('/root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini')

if config_path.exists():
    config.read(config_path)
    CLICKHOUSE_CONFIG = {
        'host': config['clickhouse']['host'],
        'port': int(config['clickhouse']['port']),
        'database': config['clickhouse']['database'],
        'user': config['clickhouse']['user'],
        'password': config['clickhouse']['password']
    }
    print("✅ Configuration loaded from clickhouse_config.ini")
else:
    # Fallback to hardcoded config
    CLICKHOUSE_CONFIG = {
        'host': 'localhost',
        'port': 9000,
        'database': 'public',
        'user': 'default',
        'password': 'DfsTeChB1'
    }
    print("⚠️  Config file not found, using default configuration")

print("\n🔌 Connecting to ClickHouse...")
print(f"📍 Host: {CLICKHOUSE_CONFIG['host']}:{CLICKHOUSE_CONFIG['port']}")
print(f"🗄️  Database: {CLICKHOUSE_CONFIG['database']}")

# Create ClickHouse client
try:
    clickhouse_client = Client(
        host=CLICKHOUSE_CONFIG['host'],
        port=CLICKHOUSE_CONFIG['port'],
        database=CLICKHOUSE_CONFIG['database'],
        user=CLICKHOUSE_CONFIG['user'],
        password=CLICKHOUSE_CONFIG['password'],
        settings={
            'max_execution_time': 7200,  # 2 hour timeout for batch processing
            'send_timeout': 600,
            'receive_timeout': 600,
            'connect_timeout': 10
        }
    )
    
    # Test connection
    result = clickhouse_client.execute('SELECT version()')
    clickhouse_version = result[0][0]
    
    print(f"\n✅ ClickHouse connection successful!")
    print(f"📦 ClickHouse Version: {clickhouse_version}")
    
    # Show available databases
    databases = clickhouse_client.execute('SHOW DATABASES')
    print(f"🗂️  Available Databases: {[db[0] for db in databases]}")
    
    # Show tables in current database
    tables = clickhouse_client.execute(f'SHOW TABLES FROM {CLICKHOUSE_CONFIG["database"]}')
    if tables:
        table_names = [tbl[0] for tbl in tables]
        print(f"📊 Tables in '{CLICKHOUSE_CONFIG['database']}': {len(table_names)} tables found")
        
        # Check for required tables
        required_tables = ['stixor_iar_distributed', 'ac_from_features_distributed']
        for req_table in required_tables:
            if req_table in table_names:
                print(f"   ✓ {req_table}")
            else:
                print(f"   ✗ {req_table} (not found)")
    else:
        print(f"📊 No tables found in '{CLICKHOUSE_CONFIG['database']}'")
    
except Exception as e:
    print(f"\n❌ Failed to connect to ClickHouse: {str(e)}")
    print("\n💡 Troubleshooting Tips:")
    print("   • Ensure ClickHouse server is running")
    print("   • Check if port 9000 is accessible")
    print("   • Verify credentials and permissions")
    print("   • Check config file at: /root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini")
    raise

print("\n" + "="*80)

✅ Configuration loaded from clickhouse_config.ini

🔌 Connecting to ClickHouse...
📍 Host: localhost:9000
🗄️  Database: public

✅ ClickHouse connection successful!
📦 ClickHouse Version: 25.10.1.3796
🗂️  Available Databases: ['INFORMATION_SCHEMA', 'default', 'information_schema', 'public', 'system']
📊 Tables in 'public': 21 tables found
   ✓ stixor_iar_distributed
   ✓ ac_from_features_distributed



In [2]:
def execute_clickhouse_query(query, return_df=False):
    """
    Execute a ClickHouse query and optionally return results as DataFrame
    
    Args:
        query: SQL query string
        return_df: If True, return results as pandas DataFrame
    
    Returns:
        Query results or None
    """
    try:
        print(f"🔄 Executing query...")
        result = clickhouse_client.execute(query, with_column_types=True)
        
        if return_df and result:
            # Extract data and column info
            data = result[0] if isinstance(result, tuple) else result
            
            if isinstance(result, tuple) and len(result) > 1:
                # Has column type information
                columns = [col[0] for col in result[1]]
                df = pd.DataFrame(data, columns=columns)
            else:
                df = pd.DataFrame(data)
            
            print(f"✅ Query executed successfully! Rows: {len(df):,}")
            return df
        else:
            print(f"✅ Query executed successfully!")
            return result
            
    except Exception as e:
        print(f"❌ Query execution failed: {str(e)}")
        raise

In [3]:
from datetime import datetime, timedelta
import time
from datetime import date

# Generate all dates in June and July 2025, excluding June 1, 5, 10, 15
dates_to_process = []
start = date(2025, 8, 1)
end = date(2025, 9, 30)  # March has 30 days

delta = end - start
for i in range(delta.days + 1):
    d = start + timedelta(days=i)
    dates_to_process.append(d.strftime('%Y-%m-%d'))

print(f"📅 Total dates to process: {len(dates_to_process)}")
print(f"First date: {dates_to_process[0]}")
print(f"Last date: {dates_to_process[-1]}")

print(f"\nStarting batch processing...")

total_start_time = time.time()
successful_dates = []
failed_dates = []

for idx, CUTOFF_DATE in enumerate(dates_to_process, 1):
    print(f"\n{'='*80}")
    print(f"Processing {idx}/{len(dates_to_process)}: {CUTOFF_DATE}")
    print(f"{'='*80}")
    
    try:
        # Configuration for transaction-level features
        # Use global CUTOFF_DATE - no need to redefine
        TXN_DATE = CUTOFF_DATE  # Use the same cutoff date as user features
        
        print("📅 Transaction-Level Feature Engineering Configuration:")
        print(f"   • Transaction Date: {TXN_DATE} (from Global Cutoff Date)")
        print(f"   • Processing: 1 DAY of transactions ({TXN_DATE})")
        print(f"   • Historical Lookback: 3 days (excluding current day)")
        print(f"   • Feature Types: Transaction attributes, time-based, balance, 3d historical, one-hot encodings")
        print(f"\n💡 Note: cutoff_date column will be added to align with user features for easy filtering")
        
        # Transaction-level feature engineering query with 3-day historical features
        transaction_feature_query = f"""
INSERT INTO public.transaction_features_distributed
SELECT 
    -- Identifiers
    curr.trans_id,
    curr.ac_from,
    curr.ac_to,
    curr.data_date,
    curr.trans_initiate_time,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- Original transaction attributes
    curr.trx_channel,
    curr.trx_type,
    curr.start_balance,
    curr.end_balance,
    curr.trx_amt,
    
    -- Time-based features
    toHour(curr.trans_initiate_time) as hour_of_day,
    toDayOfWeek(curr.trans_initiate_time) as day_of_week,
    if(toDayOfWeek(curr.trans_initiate_time) IN (6, 7), 1, 0) as is_weekend,
    if(toHour(curr.trans_initiate_time) >= 22 OR toHour(curr.trans_initiate_time) <= 6, 1, 0) as is_night,
    if(toHour(curr.trans_initiate_time) >= 9 AND toHour(curr.trans_initiate_time) <= 17, 1, 0) as is_business_hours,
    if(toHour(curr.trans_initiate_time) < 6 OR toHour(curr.trans_initiate_time) > 23, 1, 0) as is_unusual_hour,
    
    -- Risk indicators (current transaction)
    if((toHour(curr.trans_initiate_time) >= 22 OR toHour(curr.trans_initiate_time) <= 6) 
       AND toDayOfWeek(curr.trans_initiate_time) IN (6, 7), 1, 0) as night_weekend_combo,
    
    -- Balance features
    log(greatest(curr.start_balance, 1)) as start_balance_log,
    curr.end_balance - curr.start_balance as balance_change,
    if(curr.start_balance > 0, (curr.end_balance - curr.start_balance) / curr.start_balance, 0) as balance_change_pct,
    
    -- 3-day historical features (excluding current transaction day)
    coalesce(hist.txns_3d, 0) as txns_3d,
    coalesce(hist.total_amount_3d, 0) as total_amount_3d,
    coalesce(hist.avg_amount_3d, 0) as avg_amount_3d,
    coalesce(hist.max_amount_3d, 0) as max_amount_3d,
    coalesce(hist.min_amount_3d, 0) as min_amount_3d,
    coalesce(hist.unique_recipients_3d, 0) as unique_recipients_3d,
    coalesce(hist.unique_channels_3d, 0) as unique_channels_3d,
    coalesce(hist.unique_types_3d, 0) as unique_types_3d,
    if(coalesce(hist.txns_3d, 0) > 10, 1, 0) as is_high_activity_3d,
    if(coalesce(hist.unique_channels_3d, 0) > 1, 1, 0) as multi_channel_recent,
    if(coalesce(hist.avg_amount_3d, 0) > 0, 
       (curr.trx_amt - hist.avg_amount_3d) / hist.avg_amount_3d, 0) as amount_deviation_from_avg,
    coalesce(hist.night_txns_3d, 0) as night_txns_3d,
    coalesce(hist.weekend_txns_3d, 0) as weekend_txns_3d,
    
    -- Channel one-hot encoding
    if(curr.trx_channel = 'NEW_JC_APP', 1, 0) as channel_new_jc_app,
    if(curr.trx_channel = 'USSD', 1, 0) as channel_ussd,
    if(curr.trx_channel = 'USSD_API', 1, 0) as channel_ussd_api,
    if(curr.trx_channel = 'Payment Gateway', 1, 0) as channel_payment_gateway,
    if(curr.trx_channel = 'Mobile App', 1, 0) as channel_mobile_app,
    
    -- Type one-hot encoding
    if(curr.trx_type = 'Transfer(C2C)', 1, 0) as type_transfer_c2c,
    if(curr.trx_type = 'Transfer(C2B)', 1, 0) as type_transfer_c2b,
    if(curr.trx_type = 'Bill Payment', 1, 0) as type_bill_payment,
    if(curr.trx_type LIKE '%Load%', 1, 0) as type_mobile_load,
    
    -- Metadata
    now() as processing_timestamp,
    today() as created_at

FROM (
    SELECT *
    FROM public.stixor_iar_distributed
    WHERE data_date = toDate('{TXN_DATE}')
      AND ac_from != ''
) AS curr

GLOBAL LEFT JOIN (
    SELECT 
        ac_from,
        count() as txns_3d,
        sum(trx_amt) as total_amount_3d,
        avg(trx_amt) as avg_amount_3d,
        max(trx_amt) as max_amount_3d,
        min(trx_amt) as min_amount_3d,
        uniq(ac_to) as unique_recipients_3d,
        uniq(trx_channel) as unique_channels_3d,
        uniq(trx_type) as unique_types_3d,
        sumIf(1, toHour(trans_initiate_time) >= 22 OR toHour(trans_initiate_time) <= 6) as night_txns_3d,
        sumIf(1, toDayOfWeek(trans_initiate_time) IN (6, 7)) as weekend_txns_3d
    FROM public.stixor_iar_distributed
    WHERE data_date >= toDate('{TXN_DATE}') - INTERVAL 3 DAY
      AND data_date < toDate('{TXN_DATE}')
      AND ac_from != ''
    GROUP BY ac_from
) AS hist ON curr.ac_from = hist.ac_from
"""
        
        print("\n📝 Transaction-level feature engineering query prepared")
        print(f"   • Query length: {len(transaction_feature_query):,} characters")
        print(f"   • Target table: public.transaction_features_distributed")
        print(f"   • Date: {TXN_DATE}")
        print(f"   • Cutoff Date: {CUTOFF_DATE}")
        print(f"   • Features: 6 current + 13 historical (3d lookback) + 20 encodings = 39 features")
        print(f"   • Historical window: Excludes current day (days -3 to -1)")
        print(f"   • Includes cutoff_date column for filtering")
        
        start_time = time.time()
        
        # Execute the INSERT query
        clickhouse_client.execute(transaction_feature_query)
        
        elapsed_time = time.time() - start_time
        
        print(f"\n✅ Transaction feature engineering completed successfully!")
        print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
        
        successful_dates.append(CUTOFF_DATE)
        
    except Exception as e:
        elapsed_time = time.time() - start_time if 'start_time' in locals() else 0
        print(f"\n❌ Transaction feature engineering failed after {elapsed_time:.2f} seconds")
        print(f"   Error: {str(e)}")
        failed_dates.append((CUTOFF_DATE, str(e)))

# Summary
total_elapsed_time = time.time() - total_start_time
print(f"\n{'='*80}")
print(f"BATCH PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"⏱️  Total execution time: {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")
print(f"✅ Successful: {len(successful_dates)}/{len(dates_to_process)}")
print(f"❌ Failed: {len(failed_dates)}/{len(dates_to_process)}")

if failed_dates:
    print(f"\nFailed dates:")
    for date, error in failed_dates:
        print(f"   - {date}: {error[:100]}...")
        
# Get final count
try:
    count_query = "SELECT count(*) FROM public.transaction_features_distributed"
    total_transactions = clickhouse_client.execute(count_query)[0][0]
    print(f"\n📊 Total records in table: {total_transactions:,}")
except Exception as e:
    print(f"\n⚠️  Could not retrieve final count: {str(e)}")

📅 Total dates to process: 61
First date: 2025-08-01
Last date: 2025-09-30

Starting batch processing...

Processing 1/61: 2025-08-01
📅 Transaction-Level Feature Engineering Configuration:
   • Transaction Date: 2025-08-01 (from Global Cutoff Date)
   • Processing: 1 DAY of transactions (2025-08-01)
   • Historical Lookback: 3 days (excluding current day)
   • Feature Types: Transaction attributes, time-based, balance, 3d historical, one-hot encodings

💡 Note: cutoff_date column will be added to align with user features for easy filtering

📝 Transaction-level feature engineering query prepared
   • Query length: 4,043 characters
   • Target table: public.transaction_features_distributed
   • Date: 2025-08-01
   • Cutoff Date: 2025-08-01
   • Features: 6 current + 13 historical (3d lookback) + 20 encodings = 39 features
   • Historical window: Excludes current day (days -3 to -1)
   • Includes cutoff_date column for filtering

✅ Transaction feature engineering completed successfully!
⏱️ 